In [1]:
import matplotlib as plt
import numpy as np
import sage.all as sage
from sage.all import sin, cos, pi, ln, e
from Helpers.ic_generator import generate_ic_grid, get_ic_grid_info
from Helpers.state_helpers import StateAccessor
from Helpers.ode_viewer_nD import ODESystemND, ODEViewerND

In [20]:
# CONFIGURATION

N_VARS = 3  # Number of variables (not including t)

# --- Numerical Settings ---
t_start = -15.0  # Start time
t_end = 25.0  # End time
num_points = 1000  # Number of points for trajectory (higher = smoother)
step_size = 0.01  # Integration step size

max_boundary = 30 # Maximum axis limit for plot boundaries (None = auto)
compute_boundary = 2 * max_boundary if max_boundary else None  # Stop computation when values exceed this

# --- Projection Settings ---
# Which 3 variables to use as (x_axis, y_axis, z_axis) in the 3D plot
# 0 = time, 1 = first var, 2 = second var, 3 = third var, ...
projection_axes = [1, 2, 3]  # Works for any N_VARS (len MUST equal 3)
color_by = 0  # Which variable to color by (0 = t, 1 = x, 2 = y, ...) or None

# --- Initial Conitions (automatically scales with N_VARS) ---
# Format is always [x, y, z, ...] where t = 0
IC_CENTER = [1.0, 0.0, 0.0]  # Center of initial condition grid
IC_SPREAD = [2, 2, 2]  # Spread around center (by unit)
ICS_PER_VAR = [1, 1, 1]  # Number of ICs along each dimension

# --- Display Settings ---
figsize = (7, 7)  # Figure size (width, height)
title = "Phase Space"  # Plot title

# --- Output Settings ---
save_plot = True  # Save the plot to a file
save_path = "./Plots/3D/3D_NSYS_ODE.png"

# --- Live Viewer Settings ---
use_threejs = True
threejs_filename = "./Plots/3D/3D_NSYS_ODE.html"

# --- Color Settings ---
# Options: "viridis", "plasma", "inferno", "coolwarm", "RdYlBu", "twilight", etc.
colormap_name = "viridis"  # Colormap for trajectories and color bar

In [21]:
# SYSTEM DEFINITION

# Example: Each variable's derivative is sine of the next variable
# s = StateAccessor(state, VAR_NAMES).to_list()
#
# derivs = []
# for i in range(N_VARS):
#     next_idx = (i + 1) % N_VARS
#     derivs.append(sin(s[i]))
# return derivs

# Example: Lorenz Attractor
# def system_func(t, state):
#         s = StateAccessor(state, VAR_NAMES)

#         p = 10
#         r = 28
#         b = 8 / 3

#         dx = p * (s.x_1 - s.x)
#         dy = s.x * (r - s.x_2) - s.x_1
#         dz = s.x * s.x_1 - b * s.x_2

#         return [dx, dy, dz]

first_order_system = True  # Set to True for first-order systems like Lorenz
ode_order = N_VARS if not first_order_system else None

if first_order_system:
    # First-order system: define dx/dt for each variable
    VAR_NAMES = [
        f"x_{i}" if i > 0 else "x" for i in range(N_VARS)
    ]  # x_1, x_2, x_3, ...

    def system_func(t, state):
        s = StateAccessor(state, VAR_NAMES).to_list()

        derivs = []
        for i in range(N_VARS):
            n = (s[i] / t) - (t / s[i])
            derivs.append(n if abs(n) > 0.01 else 0)
        return derivs

else:
    # Higher-order system: define dⁿx/dtⁿ
    VAR_NAMES = ["x"] + [
        ("d" * i) + "x" for i in range(1, ode_order)
    ]  # [x, dx, ddx, ...]

    def highest_derivative(t, state):
        # state = [x, dx/dt, d²x/dt², ..., dⁿ⁻¹x/dtⁿ⁻¹]
        s = StateAccessor(state, VAR_NAMES)

        a = 1.0
        b = 1.0
        c = 1.0

        return 1 - 2 * (a * s.x + b * s.dx + c * s.ddx)

In [22]:
# GENERATE INITIAL CONDITIONS

ic_grid = generate_ic_grid(N_VARS, IC_CENTER, IC_SPREAD, ICS_PER_VAR)
total_ics = get_ic_grid_info(N_VARS, ICS_PER_VAR)

print(f"System: {N_VARS}-variable system")
print(f"Variables: {VAR_NAMES}")
print(f"Initial conditions: {total_ics} trajectories")
print(f"IC grid: {len(ic_grid)} total points")
print(f"Time span: [{t_start}, {t_end}]")

System: 3-variable system
Variables: ['x', 'x_1', 'x_2']
Initial conditions: 1 trajectories
IC grid: 1 total points
Time span: [-15.0, 25.0]


In [ ]:
# RUNNING AND PLOTTING

if first_order_system:
    system = ODESystemND(system_func, N_VARS, VAR_NAMES)
else:
    system = ODESystemND.from_higher_order(highest_derivative, ode_order, VAR_NAMES)

t_eval = np.linspace(t_start, t_end, num_points)
viewer = ODEViewerND()

viewer.solve_ics_grid(
    system,
    (t_start, t_end),
    ic_grid,
    total_ics,
    t_eval=t_eval,
)

print(f"✓ Solved {len(viewer.solutions)} trajectories")

viewer.plot_all_3d(
    projection_axes=projection_axes,
    color_variable=color_by,
    figsize=figsize,
    title=title,
    save_path=save_path if save_plot else None,
    max_boundary=max_boundary,
    compute_boundary=compute_boundary,
)

print(f"✓ Plotted phase space with projection axes {projection_axes}")

In [ ]:
# THREE.JS INTERACTIVE VIEWER

if use_threejs:
    cmap = plt.colormaps.get_cmap(colormap_name)
    
    def norm_to_rgb(norm_val):
        rgba = cmap(norm_val)
        return tuple(rgba[:3])  # Return (R, G, B) as 0-1 floats
    
    def rgb_to_hex(rgb):
        r, g, b = [int(round(x * 255)) for x in rgb]
        return f"#{r:02x}{g:02x}{b:02x}"
    
    traj_data = viewer.get_threejs_trajectories(
        projection_axes=projection_axes,
        color_variable=color_by,
        compute_boundary=compute_boundary,
    )

    plot_obj = sage.Graphics()

    all_color_values = []
    if color_by is not None:
        for traj in traj_data:
            if traj.get("color_values"):
                all_color_values.extend(traj["color_values"])

    for idx, traj in enumerate(traj_data):
        points = traj["points"]

        if color_by is not None and traj.get("normalized_colors") is not None:
            colors = traj["normalized_colors"]
            for i in range(0, len(points) - 1):
                color_val = colors[i]
                rgb = norm_to_rgb(color_val)
                segment = sage.line3d(
                    [points[i], points[i + 1]],
                    color=rgb_to_hex(rgb),
                    thickness=2,
                    opacity=0.8,
                )
                plot_obj += segment
        else:
            traj_color = traj.get("trajectory_color", 0)
            rgb = norm_to_rgb(traj_color)
            line = sage.line3d(
                points,
                color=rgb_to_hex(rgb),
                thickness=2,
                opacity=0.8,
            )
            plot_obj += line
    
    if color_by is not None and all_color_values:
        c_min = min(all_color_values)
        c_max = max(all_color_values)
        color_label = VAR_NAMES[color_by - 1] if 0 < color_by <= N_VARS else 'time'
        
        pos_base = max_boundary if max_boundary else 20
        bar_x = pos_base
        bar_y = pos_base
        bar_z_start = -max_boundary // 2 if max_boundary else -10
        bar_z_end = max_boundary // 2 if max_boundary else 10
        bar_length = bar_z_end - bar_z_start
        
        n_segments = 20
        for i in range(n_segments):
            norm_val = i / (n_segments - 1) if n_segments > 1 else 0
            z_pos = bar_z_start + norm_val * bar_length
            z_next = bar_z_start + ((i + 1) / (n_segments - 1)) * bar_length if i < n_segments - 1 else bar_z_end
            
            rgb = norm_to_rgb(norm_val)
            segment = sage.line3d(
                [[bar_x, bar_y, z_pos], [bar_x, bar_y, z_next]],
                color=rgb_to_hex(rgb),
                thickness=10,
            )
            plot_obj += segment
        
        plot_obj += sage.text3d(color_label, (bar_x + (1 / max_boundary), bar_y, bar_z_end - bar_length / 2), fontsize=12, color='black')
        plot_obj += sage.text3d(f"{c_max:.2f}", (bar_x + (1 / max_boundary), bar_y, bar_z_end), fontsize=10, color='black')
        plot_obj += sage.text3d(f"{c_min:.2f}", (bar_x + (1 / max_boundary), bar_y, bar_z_start), fontsize=10, color='black')
        
        print(f"Color by: {color_by} ({color_label}), range: [{c_min:.2f}, {c_max:.2f}]")
        print(f"Colormap: {colormap_name}")
    elif color_by is not None:
        print(f"Color by: {color_by} ({VAR_NAMES[color_by-1] if 0 < color_by <= N_VARS else 'time'})")

    plot_obj.save(threejs_filename, viewer="threejs", online=True)
    print(f"✓ Three.js plot saved to: {threejs_filename}")
    sage.show(plot_obj)
else:
    print("Three.js viewer not set to True")

Completed trajectory 1 of 1
Color by: 0 (time), range: [-15.00, 25.00]
Colormap: viridis
✓ Three.js plot saved to: ./Plots/3D/3D_NSYS_ODE.html
Graphics3d Object
